# HPT prototype: 600-case eval on Colab (A100, no UI)

Runs `pt_healthcare.eval_runner.evaluate()` with in-process vLLM and writes one CSV row per eval case. Run the cells in order in a **fresh A100 runtime**.

Before you start:
1. Add a Colab secret named `HF_TOKEN` (key icon, left sidebar) with notebook access on. Your Hugging Face account must have accepted the `google/medgemma-1.5-4b-it` license.
2. Put `everyUPMCmrf_clean.csv` in Google Drive and set `MRF_CSV` below.

In [ ]:
# 1. Settings: the only cell you need to edit
REPO_URL  = "https://github.com/adi274903/HPT_Prototype"
REPO_DIR  = "/content/HPT_Prototype"
MRF_CSV   = "/content/drive/MyDrive/everyUPMCmrf_clean.csv"   # check the exact filename in Drive
MODEL     = "google/medgemma-1.5-4b-it"
OUT_DIR   = "/content/drive/MyDrive/hpt_eval"                 # results are saved here
# Which eval set to score:
#   "eval_dataset" or "eval"          -> a folder in the repo (relative to REPO_DIR)
#   "/content/drive/.../A1_dataset.csv" -> a flat CSV with columns Query, CPT, Hospital, Payer, Plan, Negotiated_rate
#   an absolute folder path            -> a folder holding pt_eval_v2.jsonl + pt_eval_v2_targets.jsonl
DATASET_DIR = "eval_dataset"
PER_SUBTYPE = 0      # 0 = all 600 cases; 1 = a 10-case smoke test

In [ ]:
# 2. Clone the repo and install (vllm swaps torch, so keep this a fresh runtime)
import os
if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git pull -q
!pip install -q -r requirements-eval.txt

In [ ]:
# 3. Google Drive
from google.colab import drive
drive.mount("/content/drive")
assert os.path.isfile(MRF_CSV), f"MRF not found: {MRF_CSV}. List the folder with !ls to get the exact name."
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# 4. Hugging Face login: environment first, then Colab secrets
from huggingface_hub import login

NAMES = ("HF_TOKEN", "HUGGINGFACE_API_KEY", "HUGGINGFACE_HUB_TOKEN")
token = next((os.getenv(n) for n in NAMES if os.getenv(n)), None)
if not token:
    from google.colab import userdata
    for name in NAMES:
        try:
            token = userdata.get(name)
        except Exception:
            token = None
        if token:
            break
assert token, "No Hugging Face token: add a Colab secret named HF_TOKEN (notebook access on) or export HF_TOKEN."
os.environ["HF_TOKEN"] = token
login(token)
print("Logged in to Hugging Face.")

In [ ]:
# 5. GPU check
import torch
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > A100"
print(torch.cuda.get_device_name(0))

In [ ]:
# 6. Price database, once per runtime (~8 min over the 2.3 GB CSV)
DB = ".cache/mrf_eval.duckdb"
if not os.path.exists(DB):
    !python scripts/prepare_eval_mrf.py --mrf "{MRF_CSV}" --out {DB}
assert os.path.exists(DB), "prepare_eval_mrf.py failed, read its output above"

In [ ]:
# 7. Evaluate. Each finished case is appended to eval_run_<dataset>.jsonl, so a disconnect resumes.
#    Use fresh=True to start over.
from pathlib import Path
from pt_healthcare.eval_runner import evaluate, format_report

DATASET = Path(DATASET_DIR) if os.path.isabs(DATASET_DIR) else Path(REPO_DIR) / DATASET_DIR
TAG = DATASET.stem                                  # keeps each dataset's results apart
RUN_JSONL, REPORT_JSON, RESULTS_CSV = f"eval_run_{TAG}.jsonl", f"eval_report_{TAG}.json", f"eval_results_{TAG}.csv"
print("dataset:", DATASET)

report = evaluate(
    DATASET,
    out=RUN_JSONL,
    report_path=REPORT_JSON,
    backend="vllm",
    model=MODEL,
    dtype="bfloat16",                # A100 supports it
    per_subtype=PER_SUBTYPE,
    max_model_len=8192,
    gpu_memory_utilization=0.90,
    db=DB,
)
print(format_report(report))
print(report["codes"]["dice_mrf"], report["rows"]["dice_raw"])

In [ ]:
# 8. One CSV row per case: query, predictions, scores, latency, errors
import json, pandas as pd
from pt_healthcare.eval_runner import find_dataset_dir, load_jsonl, CASES_FILE

records = load_jsonl(RUN_JSONL)
queries = {c["eval_id"]: c["eval_query"] for c in load_jsonl(find_dataset_dir(report["run"]["dataset"]) / CASES_FILE)}

df = pd.json_normalize(records, sep=".")
df.insert(1, "eval_query", df["eval_id"].map(queries))
# lists and dicts -> JSON text so each case stays on one CSV row
for col in df.columns:
    if df[col].map(lambda v: isinstance(v, (list, dict))).any():
        df[col] = df[col].map(lambda v: json.dumps(v, ensure_ascii=False) if isinstance(v, (list, dict)) else v)

df.to_csv(RESULTS_CSV, index=False)
print(df.shape, "->", RESULTS_CSV)
df[[c for c in ("eval_id", "subtype", "intent_pred", "codes.dice_mrf", "rows.dice_raw") if c in df.columns]].head()

In [ ]:
!cp "{RESULTS_CSV}" "{RUN_JSONL}" "{REPORT_JSON}" "{OUT_DIR}/"
!ls -la "{OUT_DIR}"